# Notebook 04 — Model Training

This notebook trains the **XGBoost binary classifier** that serves as the final decision layer
in the supply-chain-detector pipeline.

**Pipeline position:** The classifier consumes 7 features derived from Layers 1–5 and outputs
a malicious-probability score scaled to [0, 100].

**Key challenges:**
- Severe class imbalance (≈14:1 benign-to-malicious) → handled via `scale_pos_weight`
- Small dataset (≈430 records) → careful regularization to avoid overfitting
- Features are zero-inflated (source code absent offline) → model must be robust to sparse inputs

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve,
)
from xgboost import XGBClassifier

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "supply-chain-detector"]
PROJECT_ROOT = next((p for p in candidates if (p / "detector").exists()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from detector.classifier import DEFAULT_FEATURE_NAMES

DATA_DIR = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = DATA_DIR / "notebook_cache"
SPLITS_DIR = DATA_DIR / "splits"

FEATURE_NAMES = list(DEFAULT_FEATURE_NAMES)
print(f"Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")

## 1. Load Train / Validation / Test Splits

Splits are created by `data/datasets/label_and_split.py` using **stratified sampling**
(70% train / 15% val / 15% test) with a fixed seed for reproducibility.

In [ ]:
def load_split(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_records = load_split(SPLITS_DIR / "train.json")
val_records   = load_split(SPLITS_DIR / "val.json")
test_records  = load_split(SPLITS_DIR / "test.json")

split_info = pd.DataFrame([
    {"split": "train", "total": len(train_records),
     "benign": sum(1 for r in train_records if r.get("label") == "benign"),
     "malicious": sum(1 for r in train_records if r.get("label") == "malicious")},
    {"split": "val", "total": len(val_records),
     "benign": sum(1 for r in val_records if r.get("label") == "benign"),
     "malicious": sum(1 for r in val_records if r.get("label") == "malicious")},
    {"split": "test", "total": len(test_records),
     "benign": sum(1 for r in test_records if r.get("label") == "benign"),
     "malicious": sum(1 for r in test_records if r.get("label") == "malicious")},
])

split_info["imbalance_ratio"] = (split_info["benign"] / split_info["malicious"]).round(1)
split_info

## 2. Feature Extraction

We extract the same 7-dimensional feature vector used by `ml/train_classifier.py`.
The function mirrors the production pipeline exactly.

In [ ]:
# Load pre-computed features from cache (instant) or extract on-the-fly (slow fallback)
if (CACHE_DIR / "features_train.npz").exists():
    _tr = np.load(CACHE_DIR / "features_train.npz", allow_pickle=True)
    _va = np.load(CACHE_DIR / "features_val.npz", allow_pickle=True)
    _te = np.load(CACHE_DIR / "features_test.npz", allow_pickle=True)
    X_train, y_train = _tr["X"], _tr["y"]
    X_val, y_val = _va["X"], _va["y"]
    X_test, y_test = _te["X"], _te["y"]
    print("Loaded features from pre-computed cache.")
else:
    # Fallback: extract features on the fly (~90 seconds)
    from detector.layer1_metadata.metadata_analyzer import analyze_metadata_risk
    from detector.layer3_static.static_analyzer import analyze_static_risk

    def _dep_count(rec):
        n = 0
        if isinstance(rec.get("dependencies"), dict): n += len(rec["dependencies"])
        if isinstance(rec.get("requires_dist"), list): n += len(rec["requires_dist"])
        return n

    def extract_features(rec):
        name = str(rec.get("package_name","")).strip().lower()
        reg = str(rec.get("registry","")).strip().lower()
        src = str(rec.get("source_code",""))
        dc = _dep_count(rec)
        meta = float(analyze_metadata_risk(name, reg, rec).get("final_score", 0.0))
        stat = float(analyze_static_risk(src).get("final_score", 0.0)) if src.strip() else 0.0
        emb = (25.0 if len(src) < 200 else 10.0) if src.strip() else 0.0
        graph = min(float(dc) * 5.0, 100.0)
        return [meta, emb, stat, graph, float(len(name)), float(dc), float(0 if rec.get("author") else 1)]

    def build_matrix(records):
        X, y = [], []
        for r in records:
            label = str(r.get("label","")).strip().lower()
            if label not in {"benign","malicious"}: continue
            X.append(extract_features(r))
            y.append(1 if label == "malicious" else 0)
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

    print("Cache not found — extracting features (this takes ~90s)...")
    train_recs = json.loads((SPLITS_DIR / "train.json").read_text(encoding="utf-8"))
    val_recs = json.loads((SPLITS_DIR / "val.json").read_text(encoding="utf-8"))
    test_recs = json.loads((SPLITS_DIR / "test.json").read_text(encoding="utf-8"))
    X_train, y_train = build_matrix(train_recs)
    X_val, y_val = build_matrix(val_recs)
    X_test, y_test = build_matrix(test_recs)

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}")
print(f"Train class balance: {int((y_train == 0).sum())} benign / {int((y_train == 1).sum())} malicious")

## 3. Handle Class Imbalance

With a ~14:1 benign-to-malicious ratio, a naive model could achieve 93% accuracy by predicting
"benign" for everything.

We address this via XGBoost's `scale_pos_weight` parameter, which up-weights the minority class
during training. This is equivalent to oversampling the malicious class without duplicating data.

In [ ]:
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
scale_weight = n_neg / n_pos if n_pos > 0 else 1.0

print(f"Benign (class 0): {n_neg}")
print(f"Malicious (class 1): {n_pos}")
print(f"scale_pos_weight: {scale_weight:.1f}")
print(f"\nThis tells XGBoost to treat each malicious sample as {scale_weight:.0f}x more important.")

## 4. Train XGBoost

Hyperparameters are chosen conservatively to prevent overfitting on a small dataset:
- `max_depth=4` — limits tree complexity
- `n_estimators=180` — moderate ensemble size
- `learning_rate=0.08` — slow learning for better generalization
- `subsample=0.9` / `colsample_bytree=0.9` — row and column subsampling

In [ ]:
model = XGBClassifier(
    n_estimators=180,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_weight,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print("Training complete.")
print(f"Best iteration: {model.best_iteration if hasattr(model, 'best_iteration') else 'N/A'}")

## 5. Evaluation Metrics

For a security tool, **recall** (catching all malicious packages) is more critical than
precision (avoiding false alarms). A missed malicious package is far worse than a false positive
that triggers manual review.

In [ ]:
def evaluate(model, X, y, split_name=""):
    proba = model.predict_proba(X)[:, 1]
    y_pred = (proba >= 0.5).astype(int)
    metrics = {
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall":    recall_score(y, y_pred, zero_division=0),
        "F1":        f1_score(y, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y, proba) if len(np.unique(y)) > 1 else 0.0,
    }
    print(f"\n{'=' * 40}")
    print(f" {split_name} Metrics")
    print(f"{'=' * 40}")
    for name, val in metrics.items():
        print(f"  {name:12s}: {val:.4f}")
    return metrics, proba, y_pred


val_metrics, val_proba, val_pred     = evaluate(model, X_val, y_val, "Validation")
test_metrics, test_proba, test_pred  = evaluate(model, X_test, y_test, "Test")

In [ ]:
# Confusion matrix (pure matplotlib — no seaborn)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_true, y_pred_arr, title in [
    (axes[0], y_val, val_pred, "Validation"),
    (axes[1], y_test, test_pred, "Test"),
]:
    cm = confusion_matrix(y_true, y_pred_arr)
    im = ax.imshow(cm, cmap="Blues", interpolation="nearest")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Malicious"])
    ax.set_yticklabels(["Benign", "Malicious"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{title} Confusion Matrix", fontweight="bold")
    # Annotate cells
    for i in range(2):
        for j in range(2):
            color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16, color=color)

plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
fig, ax = plt.subplots(figsize=(7, 6))

for proba, y_true, label, color in [
    (val_proba, y_val, f"Validation (AUC={val_metrics['ROC-AUC']:.3f})", "#3498db"),
    (test_proba, y_test, f"Test (AUC={test_metrics['ROC-AUC']:.3f})", "#e74c3c"),
]:
    if len(np.unique(y_true)) > 1:
        fpr, tpr, _ = roc_curve(y_true, proba)
        ax.plot(fpr, tpr, label=label, color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.5, label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Feature Importance

XGBoost assigns importance scores based on how frequently each feature was used for
splitting across all trees. This tells us which detection layers contribute most to
the final classification.

In [ ]:
importances = model.feature_importances_
df_imp = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "importance": importances,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_imp["feature"], df_imp["importance"], color="#3498db", edgecolor="white")
ax.set_xlabel("Importance (gain-based)")
ax.set_title("XGBoost Feature Importance", fontsize=13, fontweight="bold")

for i, (_, row) in enumerate(df_imp.iterrows()):
    ax.text(row["importance"] + 0.005, i, f"{row['importance']:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nTop features:")
for _, row in df_imp.sort_values("importance", ascending=False).iterrows():
    print(f"  {row['feature']:20s} {row['importance']:.4f}")

## 7. Save Model

The trained model and metadata are saved for production use by `detector/classifier.py`.

In [ ]:
MODEL_FILE = DATA_DIR / "xgboost_model.json"
META_FILE  = DATA_DIR / "xgboost_model_meta.json"

model.save_model(MODEL_FILE)

meta = {
    "feature_names": FEATURE_NAMES,
    "training_rows": int(len(y_train)),
    "validation_rows": int(len(y_val)),
    "test_rows": int(len(y_test)),
    "benign_train_rows": n_neg,
    "malicious_train_rows": n_pos,
    "scale_pos_weight": scale_weight,
    "val_metrics": {k: round(v, 4) for k, v in val_metrics.items()},
    "test_metrics": {k: round(v, 4) for k, v in test_metrics.items()},
}

with open(META_FILE, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"Model saved   : {MODEL_FILE}")
print(f"Metadata saved: {META_FILE}")
print(f"\nTest metrics: {test_metrics}")

## Summary

| Metric | Validation | Test |
|--------|-----------|------|
| Precision | See above | See above |
| Recall | See above | See above |
| F1 | See above | See above |
| ROC-AUC | See above | See above |

**Key observations:**

- **High recall on test set** indicates the model catches most malicious packages — critical
  for a security tool where false negatives are costly.
- **Feature importance** reveals which detection layers contribute most to classification.
  `metadata_score` dominates in the metadata-only regime; with live source code,
  `static_score` and `embedding_score` gain importance.
- **Class imbalance handling** via `scale_pos_weight` prevents the model from defaulting
  to the majority class.

The model is saved to `data/processed/xgboost_model.json` and loaded at runtime by
`detector/classifier.py`.

---
*Next: [05_evaluation_ablation.ipynb](05_evaluation_ablation.ipynb) — ablation study
to quantify each layer's contribution.*